# GC Building blocks

In [1]:
import secrets

def gen_label(num_bytes=16):
    """Generate a random label of num_bytes length."""
    return secrets.token_bytes(num_bytes)

def xor_bytes(a: bytes, b: bytes) -> bytes:
    """XOR two byte strings of equal length."""
    return bytes(x ^ y for x, y in zip(a, b))


In [2]:
# Step 1: Garbler generates random labels for inputs and output
# Labels for wire A (two labels, for A=0 and A=1)
A_label0 = gen_label()
A_label1 = gen_label()
# Labels for wire B
B_label0 = gen_label()
B_label1 = gen_label()
# Labels for output wire C
C_label0 = gen_label()
C_label1 = gen_label()

# (Optional) For demonstration, show the labels in hex format
print("Wire A labels:", A_label0.hex(), "(for 0),", A_label1.hex(), "(for 1)")
print("Wire B labels:", B_label0.hex(), "(for 0),", B_label1.hex(), "(for 1)")
print("Output wire C labels:", C_label0.hex(), "(for 0),", C_label1.hex(), "(for 1)")


Wire A labels: ae2377d332d0e30f5c869035d6434b82 (for 0), 24bbdfa46c56489040b8067b895749e5 (for 1)
Wire B labels: 24c202d3dcf750219bc28c2390b3f12a (for 0), ed4027e1cf79cf5330bc72458c15e237 (for 1)
Output wire C labels: e7ecfde885a2ae2307d6699f4fb46dc3 (for 0), f308d8920fefc2bac19dbb4f6f2c9992 (for 1)


In [3]:
import random

# Step 2: Garble the AND truth table
garbled_table_AND = []

# Encrypt output label for each input combination
# (A_val, B_val) -> corresponding output label encrypted with A_label and B_label
cipher_00 = xor_bytes(C_label0, xor_bytes(A_label0, B_label0))  # A=0, B=0 -> output 0
cipher_01 = xor_bytes(C_label0, xor_bytes(A_label0, B_label1))  # A=0, B=1 -> output 0
cipher_10 = xor_bytes(C_label0, xor_bytes(A_label1, B_label0))  # A=1, B=0 -> output 0
cipher_11 = xor_bytes(C_label1, xor_bytes(A_label1, B_label1))  # A=1, B=1 -> output 1

garbled_table_AND = [cipher_00, cipher_01, cipher_10, cipher_11]
random.shuffle(garbled_table_AND)  # shuffle the entries

print("Garbled AND table entries (hex):")
for entry in garbled_table_AND:
    print(entry.hex())


Garbled AND table entries (hex):
6d0d88e86b851d0dc09275890944d76b
a48fadda780b827f6bec8bef15e2c476
e795209f3503b692dcace3c75650d50c
3af320d7acc04579b199cf716a6e3240


In [4]:
# Simulated inputs
x = 0  # Alice's input bit for wire A
y = 1  # Bob's input bit for wire B

# Alice (Garbler) provides her input label and the garbled table to Bob.
alice_label_for_x = A_label0 if x == 0 else A_label1

# Bob (Evaluator) obtains his input label via OT (simulated here):
bob_label_for_y = B_label0 if y == 0 else B_label1

print(f"Alice's input bit: {x}, she sends label: {alice_label_for_x.hex()[:8]}... (truncated)")
print(f"Bob's input bit: {y}, he obtains label: {bob_label_for_y.hex()[:8]}... (truncated)")

# Bob now evaluates the garbled AND gate:
output_label = None
for ciphertext in garbled_table_AND:
    # Try decrypting with the two input labels
    trial = xor_bytes(ciphertext, xor_bytes(alice_label_for_x, bob_label_for_y))
    if trial in (C_label0, C_label1):
        output_label = trial
        break

# Determine the actual output bit by checking which output label it is
if output_label == C_label0:
    result_bit = 0
elif output_label == C_label1:
    result_bit = 1
else:
    result_bit = None  # this would indicate decryption failed

print("Evaluator obtained output label:", output_label.hex())
print("Decrypted AND result bit:", result_bit)


Alice's input bit: 0, she sends label: ae2377d3... (truncated)
Bob's input bit: 1, he obtains label: ed4027e1... (truncated)
Evaluator obtained output label: e7ecfde885a2ae2307d6699f4fb46dc3
Decrypted AND result bit: 0


In [5]:
# Brute-force test all input pairs for the AND gate
for a in [0, 1]:
    for b in [0, 1]:
        # Garbler picks labels (we reuse the ones already generated for consistency)
        alice_label = A_label0 if a == 0 else A_label1
        bob_label = B_label0 if b == 0 else B_label1
        # Evaluator tries to decrypt using these labels
        output_label = None
        for ciphertext in garbled_table_AND:
            trial = xor_bytes(ciphertext, xor_bytes(alice_label, bob_label))
            if trial in (C_label0, C_label1):
                output_label = trial
                break
        decoded_bit = 0 if output_label == C_label0 else (1 if output_label == C_label1 else None)
        print(f"A={a}, B={b} -> Garbled AND result: {decoded_bit}")


A=0, B=0 -> Garbled AND result: 0
A=0, B=1 -> Garbled AND result: 0
A=1, B=0 -> Garbled AND result: 0
A=1, B=1 -> Garbled AND result: 1


In [6]:
# New labels for XOR gate wires: let's name them D, E for inputs and F for output to avoid confusion with previous labels.
D_label0 = gen_label()
D_label1 = gen_label()
E_label0 = gen_label()
E_label1 = gen_label()
F_label0 = gen_label()
F_label1 = gen_label()

# Build garbled table for XOR gate
garbled_table_XOR = []
# XOR outputs: 0⊕0=0, 0⊕1=1, 1⊕0=1, 1⊕1=0
cipher_00 = xor_bytes(F_label0, xor_bytes(D_label0, E_label0))  # D=0, E=0 -> output 0
cipher_01 = xor_bytes(F_label1, xor_bytes(D_label0, E_label1))  # D=0, E=1 -> output 1
cipher_10 = xor_bytes(F_label1, xor_bytes(D_label1, E_label0))  # D=1, E=0 -> output 1
cipher_11 = xor_bytes(F_label0, xor_bytes(D_label1, E_label1))  # D=1, E=1 -> output 0

garbled_table_XOR = [cipher_00, cipher_01, cipher_10, cipher_11]
random.shuffle(garbled_table_XOR)

print("Garbled XOR table entries (hex):")
for entry in garbled_table_XOR:
    print(entry.hex())


Garbled XOR table entries (hex):
0148204a5200e25f6f9433c7486c5458
e1d62978643e121f005aaf507a544976
13bd3e08fbc616dada0debefd5d6496b
f323373acdf8e69ab5c37778e7ee5445


In [7]:
# Simulate input for XOR gate
a = 1  # value for wire D (garbler's input for XOR gate)
b = 0  # value for wire E (evaluator's input for XOR gate)

# Garbler provides D's label for a and the garbled XOR table
alice_label_for_a = D_label0 if a == 0 else D_label1
# Evaluator obtains E's label for b via OT
bob_label_for_b = E_label0 if b == 0 else E_label1

# Evaluator decrypts the garbled XOR table
output_label = None
for ciphertext in garbled_table_XOR:
    trial = xor_bytes(ciphertext, xor_bytes(alice_label_for_a, bob_label_for_b))
    if trial in (F_label0, F_label1):
        output_label = trial
        break

# Determine output bit from output label
result_bit = 0 if output_label == F_label0 else (1 if output_label == F_label1 else None)
print(f"Input A={a}, B={b} -> XOR gate output (decrypted):", result_bit)


Input A=1, B=0 -> XOR gate output (decrypted): 1


In [8]:
# Brute-force test all input pairs for the XOR gate
for a in [0, 1]:
    for b in [0, 1]:
        alice_label = D_label0 if a == 0 else D_label1
        bob_label = E_label0 if b == 0 else E_label1
        output_label = None
        for ciphertext in garbled_table_XOR:
            trial = xor_bytes(ciphertext, xor_bytes(alice_label, bob_label))
            if trial in (F_label0, F_label1):
                output_label = trial
                break
        decoded_bit = 0 if output_label == F_label0 else (1 if output_label == F_label1 else None)
        print(f"A={a}, B={b} -> Garbled XOR result: {decoded_bit}")


A=0, B=0 -> Garbled XOR result: 0
A=0, B=1 -> Garbled XOR result: 1
A=1, B=0 -> Garbled XOR result: 1
A=1, B=1 -> Garbled XOR result: 0


# BPE Primitives

In [10]:
import secrets, random, textwrap
from typing import List, Tuple, Dict


In [11]:
def gen_label(nbytes: int = 16) -> bytes:
    return secrets.token_bytes(nbytes)

def xor_bytes(a: bytes, b: bytes) -> bytes:
    return bytes(x ^ y for x, y in zip(a, b))

class Gate:
    """One AND or XOR gate, already garbled."""
    def __init__(self, typ: str,
                 left0: bytes, left1: bytes,
                 right0: bytes, right1: bytes,
                 out0: bytes, out1: bytes):
        assert typ in ("AND", "XOR")
        self.typ = typ
        # Produce a 4-row garbled table
        if typ == "XOR":
            # Free-XOR: ciphertexts are not needed, but we’ll keep the shape uniform
            self.table = []
        else:  # AND
            c00 = xor_bytes(out0, xor_bytes(left0, right0))
            c01 = xor_bytes(out0, xor_bytes(left0, right1))
            c10 = xor_bytes(out0, xor_bytes(left1, right0))
            c11 = xor_bytes(out1, xor_bytes(left1, right1))
            self.table = [c00, c01, c10, c11]
            random.shuffle(self.table)
        self.left = (left0, left1); self.right = (right0, right1)
        self.out  = (out0, out1)

    def eval(self, L: bytes, R: bytes) -> bytes:
        if self.typ == "XOR":        # free XOR (labels preserve xor-semantics)
            if   L == self.left[0] and R == self.right[0]: return self.out[0]
            elif L == self.left[0] and R == self.right[1]: return self.out[1]
            elif L == self.left[1] and R == self.right[0]: return self.out[1]
            elif L == self.left[1] and R == self.right[1]: return self.out[0]
        else:                        # AND → brute-force decrypt table rows
            for ct in self.table:
                trial = xor_bytes(ct, xor_bytes(L, R))
                if trial in self.out:
                    return trial
        raise ValueError("Bad GC eval")

# Convenience: build new XOR label pairs “for free”
def free_xor_pair(a0: bytes, a1: bytes, b0: bytes, b1: bytes) -> Tuple[bytes, bytes]:
    """Produce labels for A xor B without extra encryption."""
    o0 = xor_bytes(a0, b0)
    o1 = xor_bytes(a1, b0)          # (0⊕1) vs (1⊕0) both give 1; pick any convention
    return o0, o1


In [12]:
# Cell 3 – utility to map text → list[int] (16-bit code-points) and back
def text_to_u16_list(s: str) -> List[int]:
    return [ord(ch) for ch in s]

def u16_list_to_text(lst: List[int]) -> str:
    return ''.join(chr(x) for x in lst)


In [16]:
assert u16_list_to_text(text_to_u16_list("hello")) == "hello"[0:5]  # round-trip all languages
